In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from SOAP.utils import transgrad, compute_planet_doppler_shift, transit_durations
from SOAP.visualizer import plot_absorption_map
import SOAP
from SOAP import PHOENIX
from SOAP.fast_starspot import doppler_shift
from astropy.constants import M_earth, M_sun
from IPython.display import HTML

In [12]:
# Star-like star, rigid rotation
radius=1                # Stellar radius [solar radii]
mass=1                  # Stellar mass [solar masses]
prot=24.47              # Stellar rotation period [days]
incl=90                 # Stellar inclination [degrees] -> 90 is equator-on, 0 is pole-on
logg= 4.4               # Stellar surface gravity [cgs]
feh= 0.00               # Stellar metallicity [dex]
Teff=5777               # Stellar effective temperature [K]
ldcn=[0.5, 0.16]       # Linear and quadratic limb-darkening coefficients (must agree with λ, see below)
law=1                   # Limb-darkening law (quadratic)
start_psi=0.0           # Starting phase of the star

# Mock jupiter-like planet close to the star
Pp=3                    # Orbital period [days]
e=0.0                   # Orbital eccentricity
w=90.0                  # Argument of periastron [degrees]
ip=90                   # Orbital inclination [degrees]
lbda=0                  # Sky-projected spin-orbit misalignement [degrees]
a=9                     # Semi-major axis [stellar radii]
Rp=0.15                 # Planetary radius [stellar radii]
Mp=317.8                # Planetary mass [Earth masses]

# Observational parameters
λ =[5882, 5902]         # Wavelength range [Angstrom]
Res=140000              # Instrumental resolution

In [13]:
spQ = PHOENIX(wave_range=λ, normalize=True, teff=Teff, logg=logg, Z=feh)
spQ_active = PHOENIX(wave_range=λ, normalize=True, teff=Teff, logg=logg, Z=feh, contrast=-600, ar=True)

tplt={"wave":spQ.wave, "flux":spQ.flux} # Template spectrum matching the input local spectra to cumpute the RVs

In [14]:
sim = SOAP.Simulation(pixel=spQ, pixel_ar=spQ_active, wlll=(λ[1]+λ[0])/2, inst_reso=Res, grid=600, active_regions=[], ring=None)

In [15]:
sim.active_regions=[SOAP.ActiveRegion(check = True,
                             lon = 0,
                             lat = 0,
                             size = 0.1 ,
                             temp_diff = 600, # in K, this is the default value
                             active_region_type = 0
                            )]

In [16]:
# Set the properties of the star and planet
sim.star.set(prot=prot,incl=incl,coeffs=ldcn,law=1, start_psi=start_psi,radius=radius,mass=mass,teff=Teff)
sim.planet.Rp=0#(P=Pp,t0=start_psi,e=e,w=w,ip=ip,lbda=lbda,a=a,Rp=Rp,Mp=Mp)

In [17]:
sim.star

SOAP.Star(prot=24.47 d; incl=90.0 deg; radius=1.0 solRad; mass=1.0 solMass; teff=5777 K; diffrotB=0.0; diffrotC=0.0; cb1=0.0; law=(1, 'quadratic'); coeffs=[0.5, 0.16]; start_psi=0.0)

In [18]:
# Provide the range of stellar phases to simulate
psi = np.linspace(-0.1,0.1,50) / sim.star.prot

In [19]:
# Calculate the transit signal
out=sim.calculate_signal(psi, renormalize_rv=True, template=tplt)

In [20]:
print(sim.pixel_trans)

AttributeError: 'Simulation' object has no attribute 'pixel_trans'